# Build `oil_network` schema

Defines the full relational schema for the asset-centric oil-network model in PostgreSQL, schema `oil_network`.

**Repeatable:** the first cell drops and recreates the schema, so the entire notebook can be re-run from scratch with no leftover state. Inspect the result in DBeaver (right-click the schema → ER Diagram).

**Built incrementally**, one concept at a time.

- **Step 1** — `locations`, `assets`, `graphs`, `nodes`. Real-world identity, where things are, how assets are positioned in a graph.
- **Step 2** — `commodities`, `variable_types`, `node_types`, `variables`. The variable layer (slots) plus the node-type catalog.
- **Step 3** — `timeseries_types`, `timeseries`, `timeseries_data`. Asset-level series with vintaged values.
- **Step 4** — `scenarios`. Per-graph scenario registry.
- **Step 5** — `variable_assignments`, `node_type_default_formulas`, plus two `BEFORE INSERT/UPDATE` triggers enforcing the same-graph constraints that Postgres CHECK can't express. **Trigger functions are schema-qualified** so they work regardless of the caller's `search_path`.
- **Step 6** — Promoted columns (flat fields commonly queried) + a single `scenario_notes` table for free-form scope narrative.
- **Step 7** — Five derivation views that surface the network's flow / aggregation structure and the per-scenario authoritative / collapsed status — all derived from `variable_assignments`. Scope is no longer a separate set of stored tables; it emerges from the assignments.

## 0. Init

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

PG_HOST   = os.environ.get("PG_HOST",   "localhost")
PG_PORT   = os.environ.get("PG_PORT",   "5432")
PG_DB     = os.environ.get("PG_DB",     "eia_crude")
PG_USER   = os.environ.get("PG_USER",   "eia_user")
PG_PASS   = os.environ.get("PG_PASS",   "eia_password")
PG_SCHEMA = os.environ.get("PG_SCHEMA", "oil_network")
PG_URL    = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"

engine = create_engine(
    PG_URL,
    connect_args={"options": f"-csearch_path={PG_SCHEMA},public"},
    future=True,
)

with engine.connect() as c:
    ver, db, usr = c.execute(text("SELECT version(), current_database(), current_user")).one()
print(f"Connected:     {db} as {usr}")
print(f"Server:        {ver.splitlines()[0]}")
print(f"Target schema: {PG_SCHEMA}")

Connected:     eia_crude as eia_user
Server:        PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit
Target schema: oil_network


## 1. Reset schema

Drop and recreate `oil_network` so this notebook is idempotent — it can be re-run end-to-end and always converges to the same state.

In [2]:
with engine.begin() as c:
    c.execute(text(f"DROP SCHEMA IF EXISTS {PG_SCHEMA} CASCADE"))
    c.execute(text(f"CREATE SCHEMA {PG_SCHEMA}"))

print(f"Schema `{PG_SCHEMA}` reset.")

Schema `oil_network` reset.


## Step 1 — assets, locations, graphs, nodes

Four tables that together let us state "this asset, located here, is represented by this node in this graph."

**`locations`** — physical place. Common geographic fields are flat columns (`lat`, `lon`, `country`, `state`, `padd`, `county`, `sea`) for direct querying; everything else (e.g. coastline, basin label, climate zone) goes in `attributes` JSONB. `location_id` is an arbitrary stable text key.

**`assets`** — real-world identity. An asset has a name, optional description, a nullable `location_id` FK, and a JSONB `attributes` blob for anything else (capacity_bpd, operator, Nelson Index, sub-class metadata). Time series will eventually attach here, not to nodes — one asset, one TS, reusable across graphs.

**`graphs`** — a named asset graph. Lets us model the same assets in multiple ways (coarse vs fine, crude vs products, version 1 vs version 2).

**`nodes`** — junction. Each row is "asset X appears in graph Y as node N." `UNIQUE(graph_id, asset_id)` ensures one node per asset per graph; `node_id` is globally unique so variables and edges can reference it with a single column. No `role` column yet (deferred until we have a concrete use).

In [3]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE locations (
            location_id TEXT PRIMARY KEY,
            name        TEXT,
            lat         DOUBLE PRECISION,
            lon         DOUBLE PRECISION,
            country     TEXT,
            state       TEXT,
            padd        TEXT,
            county      TEXT,
            sea         TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))
    c.execute(text("CREATE INDEX ix_locations_state ON locations(state)"))
    c.execute(text("CREATE INDEX ix_locations_padd  ON locations(padd)"))

    c.execute(text("""
        CREATE TABLE assets (
            asset_id    TEXT PRIMARY KEY,
            name        TEXT NOT NULL,
            description TEXT,
            location_id TEXT REFERENCES locations(location_id) ON DELETE SET NULL,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
        )
    """))
    c.execute(text("CREATE INDEX ix_assets_location ON assets(location_id)"))

    c.execute(text("""
        CREATE TABLE graphs (
            graph_id    TEXT PRIMARY KEY,
            name        TEXT NOT NULL,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
        )
    """))

    c.execute(text("""
        CREATE TABLE nodes (
            node_id    TEXT PRIMARY KEY,
            graph_id   TEXT NOT NULL REFERENCES graphs(graph_id) ON DELETE CASCADE,
            asset_id   TEXT NOT NULL REFERENCES assets(asset_id),
            notes      TEXT,
            attributes JSONB NOT NULL DEFAULT '{}'::jsonb,
            UNIQUE(graph_id, asset_id)
        )
    """))
    c.execute(text("CREATE INDEX ix_nodes_graph ON nodes(graph_id)"))
    c.execute(text("CREATE INDEX ix_nodes_asset ON nodes(asset_id)"))

print("Step 1 tables created: locations, assets, graphs, nodes.")

Step 1 tables created: locations, assets, graphs, nodes.


## Step 2 — commodities, variable_types, node_types, variables

Adds the variable layer plus a `node_types` catalog so step 3's per-node-type formula defaults have a place to attach.

**`commodities`** — small lookup table. Crude, gasoline, WTI, Brent, etc. FK from `variables.commodity` (and later from `time_series.commodity`). Catches typos like `crude` vs `Crude`.

**`variable_types`** — six labels: `production`, `consumption`, `inventory`, `balancing_item`, `inflow`, `outflow`. Pure labels — no `balance_sign`, no special role for `balancing_item`. The mass-balance equation lives as a per-node formula in step 3.

**`node_types`** — small lookup. `nodes.node_type` is a nullable FK; a node without a type just doesn't pick up any default formulas, and every variable assignment must be explicit.

**`variables`** — slot definitions. One row per `(variable_type, commodity, node_id, related_node_id?)`. Two CHECKs:
- `inflow` and `outflow` must have a `related_node_id`; the other four types must not.
- `related_node_id` must differ from `node_id` (no self-edges).

Two partial UNIQUE indexes (same trick as before) handle uniqueness correctly across the nullable `related_node_id` column.

**Deferred to step 3 or later:**
- Same-graph constraint between `node_id` and `related_node_id` — needs a trigger (Postgres CHECK can't subquery cross-row).
- `node_type_default_formulas` table — pulls in step 3 alongside `variable_assignments`.

In [4]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE commodities (
            commodity   TEXT PRIMARY KEY,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    c.execute(text("""
        CREATE TABLE variable_types (
            variable_type TEXT PRIMARY KEY,
            description   TEXT,
            attributes    JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    c.execute(text("""
        CREATE TABLE node_types (
            node_type   TEXT PRIMARY KEY,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    # Add nullable node_type FK to nodes.
    c.execute(text("""
        ALTER TABLE nodes
        ADD COLUMN node_type TEXT REFERENCES node_types(node_type)
    """))
    c.execute(text("CREATE INDEX ix_nodes_type ON nodes(node_type)"))

    c.execute(text("""
        CREATE TABLE variables (
            variable_id     TEXT PRIMARY KEY,
            variable_type   TEXT NOT NULL REFERENCES variable_types(variable_type),
            commodity       TEXT NOT NULL REFERENCES commodities(commodity),
            node_id         TEXT NOT NULL REFERENCES nodes(node_id) ON DELETE CASCADE,
            related_node_id TEXT          REFERENCES nodes(node_id) ON DELETE CASCADE,
            description     TEXT,
            attributes      JSONB NOT NULL DEFAULT '{}'::jsonb,
            CHECK (related_node_id IS NULL OR related_node_id <> node_id),
            CHECK (
                (variable_type IN ('inflow','outflow') AND related_node_id IS NOT NULL)
                OR
                (variable_type NOT IN ('inflow','outflow') AND related_node_id IS NULL)
            )
        )
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX variables_uniq_nonrelational
            ON variables(variable_type, commodity, node_id)
            WHERE related_node_id IS NULL
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX variables_uniq_relational
            ON variables(variable_type, commodity, node_id, related_node_id)
            WHERE related_node_id IS NOT NULL
    """))
    c.execute(text("CREATE INDEX ix_variables_node     ON variables(node_id)"))
    c.execute(text("CREATE INDEX ix_variables_ref_node ON variables(related_node_id)"))
    c.execute(text("CREATE INDEX ix_variables_type     ON variables(variable_type)"))

print("Step 2 tables created: commodities, variable_types, node_types, variables.")
print("Column added: nodes.node_type (nullable, FK to node_types).")

Step 2 tables created: commodities, variable_types, node_types, variables.
Column added: nodes.node_type (nullable, FK to node_types).


### Seed: variable_types

Six canonical types as plain labels — no signs, no special role for `balancing_item`. Mass-balance behaviour comes from per-node formulas in step 3.

In [5]:
VARIABLE_TYPE_SEED = [
    ("production",     "Volume produced (supply injected) at the node."),
    ("consumption",    "Volume consumed at the node."),
    ("inventory",      "Stock / inventory level at the node."),
    ("balancing_item", "Residual term that absorbs reporting discrepancies. "
                       "Defined per-node via formula in step 3."),
    ("inflow",         "Per-edge inflow into this node from related_node_id."),
    ("outflow",        "Per-edge outflow from this node to related_node_id."),
]

with engine.begin() as c:
    c.execute(
        text("INSERT INTO variable_types(variable_type, description) VALUES (:t, :d)"),
        [dict(t=t, d=d) for t, d in VARIABLE_TYPE_SEED],
    )

with engine.connect() as c:
    display(pd.read_sql(text("SELECT * FROM variable_types ORDER BY variable_type"), c))

,variable_type,description,attributes
0,balancing_item,Residual term that absorbs reporting discrepan...,{}
1,consumption,Volume consumed at the node.,{}
2,inflow,Per-edge inflow into this node from related_no...,{}
3,inventory,Stock / inventory level at the node.,{}
4,outflow,Per-edge outflow from this node to related_nod...,{}
5,production,Volume produced (supply injected) at the node.,{}


## Step 3 — time series

A time series describes one measurement over time. Its subject is one or two **assets**:

- `asset_id` — the primary subject (the producing asset, the inventory holder, the source endpoint).
- `related_asset_id` — for *relational* measurements (pipeline flows, vessel voyages, inter-terminal transfers): the other endpoint. Null for non-relational TS like inventory or production at a single asset.

Both are nullable FKs to `assets` — assets are the single identity layer (physical *or* abstract; PADD 5 is just an asset whose `attributes` flag it as an aggregation). A CHECK forbids self-loops (`related_asset_id <> asset_id`).

**`timeseries_types`** — small lookup classifying *the measurement* (`production`, `consumption`, etc.). Different vocabulary from `variable_types`: TS types describe what the data measures; variable types describe a slot's role in the mass balance. Same labels at first, free to diverge later (TS could grow `forecast`, `target`, `actual`).

**`timeseries`** — the catalog. `commodity` and `timeseries_type` describe the measurement; `asset_id` (+ optional `related_asset_id`) describe the subject; `source` records who published it.

**`timeseries_data`** — the actual values. PK is `(timeseries_id, observation_date, saved_date)`, so a value revision lands as a *new row* with a later `saved_date` rather than overwriting history. Step-forward semantics: a value at `t` holds until the next observation. All values are mbd.

The link from a TS to a variable on a node happens later, in step 5's `variable_assignments`.

In [6]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE timeseries_types (
            timeseries_type TEXT PRIMARY KEY,
            description     TEXT,
            attributes      JSONB NOT NULL DEFAULT '{}'::jsonb
        )
    """))

    c.execute(text("""
        CREATE TABLE timeseries (
            timeseries_id    TEXT PRIMARY KEY,
            name             TEXT NOT NULL,
            description      TEXT,
            timeseries_type  TEXT          REFERENCES timeseries_types(timeseries_type),
            commodity        TEXT NOT NULL REFERENCES commodities(commodity),
            asset_id         TEXT          REFERENCES assets(asset_id) ON DELETE SET NULL,
            related_asset_id TEXT          REFERENCES assets(asset_id) ON DELETE SET NULL,
            source           TEXT NOT NULL,
            unit             TEXT NOT NULL DEFAULT 'kbd',
            attributes       JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at       TIMESTAMPTZ NOT NULL DEFAULT now(),
            CHECK (related_asset_id IS NULL OR related_asset_id <> asset_id)
        )
    """))
    c.execute(text("CREATE INDEX ix_ts_asset         ON timeseries(asset_id)"))
    c.execute(text("CREATE INDEX ix_ts_related_asset ON timeseries(related_asset_id)"))
    c.execute(text("CREATE INDEX ix_ts_commodity     ON timeseries(commodity)"))
    c.execute(text("CREATE INDEX ix_ts_type          ON timeseries(timeseries_type)"))
    c.execute(text("CREATE INDEX ix_ts_source        ON timeseries(source)"))
    c.execute(text("CREATE INDEX ix_ts_unit          ON timeseries(unit)"))

    c.execute(text("""
        CREATE TABLE timeseries_data (
            timeseries_id    TEXT NOT NULL REFERENCES timeseries(timeseries_id) ON DELETE CASCADE,
            observation_date DATE NOT NULL,
            saved_date       DATE NOT NULL,
            value            DOUBLE PRECISION,
            PRIMARY KEY(timeseries_id, observation_date, saved_date)
        )
    """))
    c.execute(text("CREATE INDEX ix_tsd_obsdate ON timeseries_data(observation_date)"))
    c.execute(text("CREATE INDEX ix_tsd_saved   ON timeseries_data(saved_date)"))

    # Seed timeseries_types with the same six labels as variable_types.
    # The two tables are conceptually distinct and free to diverge.
    c.execute(
        text("INSERT INTO timeseries_types(timeseries_type, description) VALUES (:t, :d)"),
        [
            {"t": "production",     "d": "Measurement of volume produced."},
            {"t": "consumption",    "d": "Measurement of volume consumed."},
            {"t": "inventory",      "d": "Measurement of stock / inventory level."},
            {"t": "balancing_item", "d": "Measurement of a balance residual."},
            {"t": "inflow",         "d": "Measurement of per-edge inflow (uses related_asset_id)."},
            {"t": "outflow",        "d": "Measurement of per-edge outflow (uses related_asset_id)."},
        ],
    )

print("Step 3 tables created: timeseries_types, timeseries, timeseries_data.")
print("  timeseries.unit (text, NOT NULL, default 'kbd') tracks the unit of each series.")
print("Seeded timeseries_types with 6 labels.")

Step 3 tables created: timeseries_types, timeseries, timeseries_data.
  timeseries.unit (text, NOT NULL, default 'kbd') tracks the unit of each series.
Seeded timeseries_types with 6 labels.


## Step 4 — scenarios

A *scenario* is a particular instantiation of a graph: which variables are assigned what data, with what formulas, over what date range. One scenario lives over one graph. We can have several scenarios on the same graph (for sensitivity analysis, for testing alternate balance closures) and several scenarios across graphs (a "coarse" scenario on the coarse graph and a "fine" scenario on the fine graph).

Step 4 just registers scenarios. The actual variable→TS-or-formula assignments happen in step 5.

In [7]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE scenarios (
            scenario_id TEXT PRIMARY KEY,
            graph_id    TEXT NOT NULL REFERENCES graphs(graph_id) ON DELETE CASCADE,
            name        TEXT NOT NULL,
            description TEXT,
            attributes  JSONB NOT NULL DEFAULT '{}'::jsonb,
            created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
        )
    """))
    c.execute(text("CREATE INDEX ix_scenarios_graph ON scenarios(graph_id)"))

print("Step 4 table created: scenarios.")

Step 4 table created: scenarios.


## Step 5 — variable_assignments, default formulas, same-graph triggers

This is the layer that turns variables into something the evaluator can compute.

**`variable_assignments`** — for `(scenario_id, variable_id, effective_from)`, says either:
- `timeseries_id` set → values come from that time series, **or**
- `formula` set → values are computed per timestep from `formula`, with `formula_inputs[]` listing the variable_ids the formula reads.

A single CHECK enforces exactly one of the two is populated: `num_nonnulls(timeseries_id, formula) = 1`. No enum kind column — the populated column *is* the kind. "Zero" is just `formula = '0'`.

`effective_from` is in the PK so a single variable can switch sources mid-run (e.g. a TS until 2020 and a formula afterwards).

**`node_type_default_formulas`** — defaults keyed on `(node_type, variable_type, commodity?)`. When a node has a `node_type` and a variable doesn't have an explicit assignment in a scenario, this is what fills in. `commodity` is nullable: NULL means "applies to every commodity at this node type." Two partial unique indexes preserve the at-most-one rule for both the NULL and non-NULL cases.

**Two triggers** for cross-row constraints Postgres CHECKs can't express:

1. `variables` → `related_node_id` must be in the same graph as `node_id`.
2. `variable_assignments` → the variable's node must live in the scenario's graph.

Both fire `BEFORE INSERT OR UPDATE`, look up the relevant `graph_id`s, and raise on mismatch.

In [8]:
with engine.begin() as c:
    c.execute(text("""
        CREATE TABLE variable_assignments (
            scenario_id     TEXT NOT NULL REFERENCES scenarios(scenario_id) ON DELETE CASCADE,
            variable_id     TEXT NOT NULL REFERENCES variables(variable_id) ON DELETE CASCADE,
            effective_from  DATE NOT NULL,
            timeseries_id   TEXT REFERENCES timeseries(timeseries_id) ON DELETE RESTRICT,
            formula         TEXT,
            formula_inputs  TEXT[] NOT NULL DEFAULT '{}'::text[],
            notes           TEXT,
            PRIMARY KEY(scenario_id, variable_id, effective_from),
            CHECK (num_nonnulls(timeseries_id, formula) = 1)
        )
    """))
    c.execute(text("CREATE INDEX ix_va_variable ON variable_assignments(variable_id)"))
    c.execute(text("CREATE INDEX ix_va_ts       ON variable_assignments(timeseries_id)"))
    c.execute(text("CREATE INDEX ix_va_scenario ON variable_assignments(scenario_id)"))

    c.execute(text("""
        CREATE TABLE node_type_default_formulas (
            node_type        TEXT NOT NULL REFERENCES node_types(node_type) ON DELETE CASCADE,
            variable_type    TEXT NOT NULL REFERENCES variable_types(variable_type),
            commodity        TEXT          REFERENCES commodities(commodity),
            default_formula  TEXT NOT NULL,
            description      TEXT
        )
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX node_type_defaults_uniq_with_commodity
            ON node_type_default_formulas(node_type, variable_type, commodity)
            WHERE commodity IS NOT NULL
    """))
    c.execute(text("""
        CREATE UNIQUE INDEX node_type_defaults_uniq_no_commodity
            ON node_type_default_formulas(node_type, variable_type)
            WHERE commodity IS NULL
    """))

    # Trigger 1: variables.related_node_id must share a graph with variables.node_id.
    # All table references inside the function body are SCHEMA-QUALIFIED so the trigger
    # works correctly regardless of the caller's search_path.
    c.execute(text("""
        CREATE OR REPLACE FUNCTION oil_network.variables_check_same_graph()
        RETURNS TRIGGER AS $$
        DECLARE
            n_graph TEXT;
            r_graph TEXT;
        BEGIN
            IF NEW.related_node_id IS NULL THEN
                RETURN NEW;
            END IF;
            SELECT graph_id INTO n_graph FROM oil_network.nodes WHERE node_id = NEW.node_id;
            SELECT graph_id INTO r_graph FROM oil_network.nodes WHERE node_id = NEW.related_node_id;
            IF n_graph IS DISTINCT FROM r_graph THEN
                RAISE EXCEPTION
                    'variables.node_id (%) is in graph % but related_node_id (%) is in graph %',
                    NEW.node_id, n_graph, NEW.related_node_id, r_graph;
            END IF;
            RETURN NEW;
        END;
        $$ LANGUAGE plpgsql
    """))
    c.execute(text("""
        CREATE TRIGGER variables_check_same_graph_trg
        BEFORE INSERT OR UPDATE ON variables
        FOR EACH ROW EXECUTE FUNCTION oil_network.variables_check_same_graph()
    """))

    # Trigger 2: variable_assignments.variable_id must live on a node in the
    # scenario's graph.  Same schema-qualification rule applies.
    c.execute(text("""
        CREATE OR REPLACE FUNCTION oil_network.variable_assignments_check_graph()
        RETURNS TRIGGER AS $$
        DECLARE
            s_graph TEXT;
            v_graph TEXT;
        BEGIN
            SELECT graph_id INTO s_graph
            FROM oil_network.scenarios WHERE scenario_id = NEW.scenario_id;
            SELECT n.graph_id INTO v_graph
            FROM oil_network.variables v JOIN oil_network.nodes n ON n.node_id = v.node_id
            WHERE v.variable_id = NEW.variable_id;
            IF s_graph IS DISTINCT FROM v_graph THEN
                RAISE EXCEPTION
                    'scenario % is over graph %, but variable % is on a node in graph %',
                    NEW.scenario_id, s_graph, NEW.variable_id, v_graph;
            END IF;
            RETURN NEW;
        END;
        $$ LANGUAGE plpgsql
    """))
    c.execute(text("""
        CREATE TRIGGER variable_assignments_check_graph_trg
        BEFORE INSERT OR UPDATE ON variable_assignments
        FOR EACH ROW EXECUTE FUNCTION oil_network.variable_assignments_check_graph()
    """))

    # scenario_node_role: per-scenario, per-node tag indicating whether a node
    # is a partition cell (balance) or an auxiliary observation / boundary (constraint)
    # in the active partition for that scenario. Only abstract aggregate nodes need
    # entries; physical nodes are implicit partition leaves.
    c.execute(text("""
        CREATE TABLE scenario_node_role (
            scenario_id TEXT NOT NULL REFERENCES scenarios(scenario_id) ON DELETE CASCADE,
            node_id     TEXT NOT NULL REFERENCES nodes(node_id)         ON DELETE CASCADE,
            role        TEXT NOT NULL CHECK (role IN ('balance', 'constraint')),
            notes       TEXT,
            PRIMARY KEY (scenario_id, node_id)
        )
    """))
    c.execute(text("CREATE INDEX ix_snr_scenario ON scenario_node_role(scenario_id)"))
    c.execute(text("CREATE INDEX ix_snr_role     ON scenario_node_role(role)"))

print("Step 5 created: variable_assignments, node_type_default_formulas, scenario_node_role, plus 2 same-graph triggers.")
print("  Trigger functions are SCHEMA-QUALIFIED so they work regardless of the caller's search_path.")

Step 5 created: variable_assignments, node_type_default_formulas, scenario_node_role, plus 2 same-graph triggers.
  Trigger functions are SCHEMA-QUALIFIED so they work regardless of the caller's search_path.


## Step 6 — Promoted columns + `scenario_notes`

Two things only:

1. **Promoted columns** on existing tables — flat fields that are commonly queried (`assets.kind`, `assets.node_class`, `assets.node_subtype`, `assets.starter_status`, `nodes.starter_status`, `scenarios.time_range_start/end/frequency/pipeline_layer_note`, `graphs.schema_version/thesis_version/source_file`, `locations.spans_multiple_padds`). These are 1-line `ALTER TABLE`s — no new tables.

2. **One narrative table** — `scenario_notes` — for the free-form text the user wants to attach to a scenario (decisions, caveats, rationale). Notes are the only piece of "scope" that genuinely doesn't derive from `variable_assignments`; everything else (authoritative levels, collapsed nodes) is now exposed through the views in Step 7.

After this step the schema has **15 tables**: 14 core + 1 narrative (`scenario_notes`). The earlier four `node_scope_*` tables (`node_scopes`, `node_scope_authorisations`, `node_scope_collapsed_nodes`, `node_scope_notes`) are gone — their content either moved to views (Step 7) or shrank into a single FK-keyed-by-scenario notes table.

In [9]:
with engine.begin() as c:
    # ---- Promote columns on existing tables ----
    c.execute(text("""
        ALTER TABLE graphs
            ADD COLUMN schema_version  TEXT,
            ADD COLUMN thesis_version  TEXT,
            ADD COLUMN source_file     TEXT
    """))
    c.execute(text("""
        ALTER TABLE locations
            ADD COLUMN spans_multiple_padds BOOLEAN NOT NULL DEFAULT FALSE
    """))
    c.execute(text("""
        ALTER TABLE assets
            ADD COLUMN node_class      TEXT,
            ADD COLUMN node_subtype    TEXT,
            ADD COLUMN kind            TEXT CHECK (kind IN ('physical','abstract')),
            ADD COLUMN starter_status  TEXT
    """))
    c.execute(text("CREATE INDEX ix_assets_kind     ON assets(kind)"))
    c.execute(text("CREATE INDEX ix_assets_subtype  ON assets(node_subtype)"))
    c.execute(text("""
        ALTER TABLE nodes
            ADD COLUMN starter_status  TEXT
    """))
    c.execute(text("""
        ALTER TABLE scenarios
            ADD COLUMN time_range_start    DATE,
            ADD COLUMN time_range_end      DATE,
            ADD COLUMN frequency           TEXT,
            ADD COLUMN pipeline_layer_note TEXT
    """))

    # ---- scenario_notes: the only piece of scope that doesn't derive ----
    # Authoritative levels and collapsed-node lists used to live in dedicated
    # tables (`node_scope_authorisations`, `node_scope_collapsed_nodes`); they
    # were redundant — every fact they encoded is now derivable from
    # `variable_assignments` and exposed via Step-7 views. The only thing left
    # is free-form narrative notes per scenario.
    c.execute(text("""
        CREATE TABLE scenario_notes (
            scenario_id TEXT NOT NULL REFERENCES scenarios(scenario_id) ON DELETE CASCADE,
            position    INT  NOT NULL,
            note_text   TEXT NOT NULL,
            PRIMARY KEY (scenario_id, position)
        )
    """))

print("Step 6 done. Promoted columns + 1 narrative table created.")
print("  ALTER: graphs (schema_version, thesis_version, source_file)")
print("  ALTER: locations (spans_multiple_padds)")
print("  ALTER: assets (node_class, node_subtype, kind, starter_status)")
print("  ALTER: nodes (starter_status)")
print("  ALTER: scenarios (time_range_start, time_range_end, frequency, pipeline_layer_note)")
print("  TABLE: scenario_notes")
print()
print("Final schema: 15 tables (14 core + 1 narrative).")

Step 6 done. Promoted columns + 1 narrative table created.
  ALTER: graphs (schema_version, thesis_version, source_file)
  ALTER: locations (spans_multiple_padds)
  ALTER: assets (node_class, node_subtype, kind, starter_status)
  ALTER: nodes (starter_status)
  ALTER: scenarios (time_range_start, time_range_end, frequency, pipeline_layer_note)
  TABLE: scenario_notes

Final schema: 15 tables (14 core + 1 narrative).


## Step 7 — Derivation views

Five read-only views that surface the graph's structure on demand. All five are derived from `variables` + `variable_assignments` — the schema's source of truth — so they automatically reflect any patch or scope change.

- **`v_flow_edges`** — the **physical flow graph**. One row per relational `outflow` variable, giving `(source, target, commodity, via_variable)`. Use this to query the actual mass-balance topology.
- **`v_aggregation_edges`** — the **aggregation graph**. One row per `(parent_variable, child_variable)` pair, derived from `formula_inputs` on every formula-bound assignment. Use this to see which variables roll up into which (and under which scenario).
- **`v_node_production_sources`** — discovery view: for every node with a `production` variable, shows its binding (observed / derived / aggregate / latent / zero), source TS or formula, and (for TS-bound) the actual date range of available data.
- **`v_scope_authoritative_nodes`** — replaces the old `node_scope_authorisations` table. One row per TS-bound `variable_assignments` entry, showing which (scenario, node, variable) is observed and via which series.
- **`v_scope_collapsed_nodes`** — replaces the old `node_scope_collapsed_nodes` table. One row per variable whose formula is `'0'` or `'latent()'` — i.e. structurally inert under the scope.

All five carry `scenario_id` where it applies, so callers filter as needed.

In [10]:
with engine.begin() as c:
    # 1. Flow graph: one row per relational outflow variable.
    #    The inflow paired-side is redundant (every outflow has a matching inflow), so we
    #    derive the flow edge from outflows only to avoid double-listing each edge.
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_flow_edges AS
        SELECT
            v.node_id          AS source,
            v.related_node_id  AS target,
            v.commodity,
            v.variable_id      AS via_variable
        FROM oil_network.variables v
        WHERE v.variable_type = 'outflow'
          AND v.related_node_id IS NOT NULL
    """))

    # 2. Aggregation graph: one row per (parent_variable, child_variable) pair,
    #    unfolded from formula_inputs. Captures every formula that depends on
    #    other variables (sum_over_children, arithmetic, single-variable refs, etc.).
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_aggregation_edges AS
        SELECT
            va.scenario_id,
            va.variable_id   AS parent_variable,
            child_var        AS child_variable,
            va.formula       AS rule
        FROM oil_network.variable_assignments va,
             unnest(va.formula_inputs) AS child_var
        WHERE cardinality(va.formula_inputs) > 0
    """))

    # 3. Node production sources: discovery view classifying each production-variable's
    #    binding kind (observed / zero / latent / aggregate / derived / unassigned),
    #    with source TS or formula, and the date range of any underlying TS facts.
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_node_production_sources AS
        WITH ts_range AS (
            SELECT
                timeseries_id,
                MIN(observation_date) AS earliest,
                MAX(observation_date) AS latest,
                COUNT(DISTINCT observation_date) AS n_observations
            FROM oil_network.timeseries_data
            GROUP BY timeseries_id
        )
        SELECT
            n.node_id,
            a.node_class,
            a.kind,
            a.starter_status,
            va.scenario_id,
            CASE
                WHEN va.timeseries_id IS NOT NULL          THEN 'observed'
                WHEN va.formula = '0'                      THEN 'zero'
                WHEN va.formula = 'latent()'               THEN 'latent'
                WHEN va.formula = 'sum_over_children'      THEN 'aggregate'
                WHEN va.formula = 'sum_over_outflows'      THEN 'sum_over_outflows'
                WHEN va.formula IS NOT NULL                THEN 'derived'
                ELSE 'unassigned'
            END                  AS binding,
            va.timeseries_id     AS source_ts,
            va.formula           AS source_formula,
            va.formula_inputs    AS depends_on,
            ts.earliest          AS data_from,
            ts.latest            AS data_to,
            ts.n_observations    AS data_points
        FROM oil_network.variables v
        JOIN oil_network.nodes  n ON n.node_id = v.node_id
        JOIN oil_network.assets a ON a.asset_id = n.asset_id
        LEFT JOIN oil_network.variable_assignments va
               ON va.variable_id = v.variable_id
        LEFT JOIN ts_range ts
               ON ts.timeseries_id = va.timeseries_id
        WHERE v.variable_type = 'production'
    """))

    # 4. Scope authoritative nodes: every TS-bound assignment is an observation
    #    on a (scenario, variable) pair. Replaces the old `node_scope_authorisations`
    #    table — the same fact is now derivable directly from variable_assignments.
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_scope_authoritative_nodes AS
        SELECT
            va.scenario_id,
            v.node_id          AS authoritative_node,
            v.variable_id,
            v.variable_type,
            v.commodity,
            va.timeseries_id   AS observed_via_ts,
            va.effective_from
        FROM oil_network.variable_assignments va
        JOIN oil_network.variables v ON v.variable_id = va.variable_id
        WHERE va.timeseries_id IS NOT NULL
    """))

    # 5. Scope collapsed nodes: every variable whose formula is '0' or 'latent()'
    #    is structurally inert under the active scope. Replaces the old
    #    `node_scope_collapsed_nodes` table.
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_scope_collapsed_nodes AS
        SELECT
            va.scenario_id,
            v.node_id          AS collapsed_node,
            v.variable_id,
            v.variable_type,
            v.commodity,
            CASE
                WHEN va.formula = '0'        THEN 'zero'
                WHEN va.formula = 'latent()' THEN 'latent'
            END                AS collapse_kind,
            va.formula
        FROM oil_network.variable_assignments va
        JOIN oil_network.variables v ON v.variable_id = va.variable_id
        WHERE va.formula IN ('0', 'latent()')
    """))

    # 6. v_inventory_changes -- for every TS-bound inventory series (unit='mbbl_level'),
    #    compute the monthly delta in MBBL and convert to kbd via per-row days-in-month.
    #    Uses the latest vintage only (saved_date DESC).
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_inventory_changes AS
        WITH latest AS (
            SELECT DISTINCT ON (td.timeseries_id, td.observation_date)
                   td.timeseries_id, td.observation_date, td.value
            FROM oil_network.timeseries_data td
            JOIN oil_network.timeseries ts USING (timeseries_id)
            WHERE ts.unit = 'mbbl_level'
            ORDER BY td.timeseries_id, td.observation_date, td.saved_date DESC
        )
        SELECT
            l.timeseries_id,
            ts.asset_id,
            ts.unit,
            l.observation_date,
            l.value                                        AS level_mbbl,
            l.value - LAG(l.value) OVER w                  AS delta_mbbl,
            (l.value - LAG(l.value) OVER w)
              / EXTRACT(DAY FROM (date_trunc('month', l.observation_date)
                                  + interval '1 month - 1 day'))::numeric
                                                           AS delta_kbd
        FROM latest l
        JOIN oil_network.timeseries ts USING (timeseries_id)
        WINDOW w AS (PARTITION BY l.timeseries_id ORDER BY l.observation_date)
    """))

    # 7. v_aggregate_balance -- pure structural pivot. Walks formula_inputs from
    #    every balancing_item variable whose formula is a real closure expression
    #    (not '0' / 'latent()'). Resolves each input variable to its TS-bound value,
    #    via direct binding OR one-level mirror dereference (when inflow = paired outflow).
    #    Pivots inputs by variable_type, computes ΔS from inventory LAG, B as residual.
    #    ZERO hardcoded series IDs / regions / CASE statements -- everything is derived
    #    from the data model.
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_aggregate_balance AS
        WITH latest AS (
            SELECT DISTINCT ON (timeseries_id, observation_date)
                   timeseries_id, observation_date, value
            FROM oil_network.timeseries_data
            ORDER BY timeseries_id, observation_date, saved_date DESC
        ),
        days AS (
            SELECT observation_date,
                   EXTRACT(DAY FROM (date_trunc('month', observation_date)
                                     + interval '1 month - 1 day'))::numeric AS d
            FROM (SELECT DISTINCT observation_date FROM latest) x
        ),
        balance_vars AS (
            SELECT v.variable_id AS b_var,
                   v.node_id     AS region_node,
                   va.formula_inputs
            FROM oil_network.variables v
            JOIN oil_network.variable_assignments va USING (variable_id)
            WHERE v.variable_type = 'balancing_item'
              AND va.formula IS NOT NULL
              AND va.formula NOT IN ('0', 'latent()')
        ),
        resolved_ts AS (
            -- Direct TS binding
            SELECT v.variable_id, va.timeseries_id
            FROM oil_network.variables v
            JOIN oil_network.variable_assignments va USING (variable_id)
            WHERE va.timeseries_id IS NOT NULL
            UNION ALL
            -- One-level mirror dereference: formula is just another variable_id
            SELECT v.variable_id, mirror_va.timeseries_id
            FROM oil_network.variables v
            JOIN oil_network.variable_assignments va USING (variable_id)
            JOIN oil_network.variable_assignments mirror_va
              ON mirror_va.variable_id = va.formula
             AND mirror_va.timeseries_id IS NOT NULL
            WHERE va.timeseries_id IS NULL
              AND va.formula IS NOT NULL
              AND va.formula NOT IN ('0', 'latent()', 'sum_over_children', 'sum_over_outflows')
              AND cardinality(va.formula_inputs) = 1
        ),
        balance_inputs AS (
            SELECT bv.region_node,
                   input_v.variable_type AS input_type,
                   td.observation_date,
                   td.value
            FROM balance_vars bv
            CROSS JOIN LATERAL unnest(bv.formula_inputs) AS u(child_var_id)
            JOIN oil_network.variables input_v ON input_v.variable_id = u.child_var_id
            JOIN resolved_ts rt                 ON rt.variable_id = u.child_var_id
            JOIN latest td                       ON td.timeseries_id = rt.timeseries_id
        ),
        pivoted AS (
            SELECT region_node, observation_date,
                   MAX(value) FILTER (WHERE input_type='production')  AS p_kbd,
                   MAX(value) FILTER (WHERE input_type='consumption') AS c_kbd,
                   MAX(value) FILTER (WHERE input_type='inventory')   AS s_mbbl,
                   SUM(value) FILTER (WHERE input_type='inflow')      AS fin_kbd,
                   SUM(value) FILTER (WHERE input_type='outflow')     AS fout_kbd
            FROM balance_inputs
            GROUP BY region_node, observation_date
        )
        SELECT
            region_node, observation_date,
            s_mbbl,
            (s_mbbl - LAG(s_mbbl) OVER w) / d.d AS ds_kbd,
            p_kbd, c_kbd, fin_kbd, fout_kbd,
            ((s_mbbl - LAG(s_mbbl) OVER w) / d.d) - p_kbd + c_kbd - fin_kbd + fout_kbd AS b_kbd
        FROM pivoted
        JOIN days d USING (observation_date)
        WINDOW w AS (PARTITION BY region_node ORDER BY observation_date)
    """))

    # 8. v_aggregation_consistency -- for every TS-bound variable that ALSO has
    #    explicit aggregation constituents declared in formula_inputs, compute
    #    sum(constituents) per observation_date and compare to the TS value.
    #    Material gap = data-inconsistency flag. The TS value is authoritative;
    #    this view is purely diagnostic.
    c.execute(text("""
        CREATE OR REPLACE VIEW oil_network.v_aggregation_consistency AS
        WITH latest AS (
            SELECT DISTINCT ON (timeseries_id, observation_date)
                   timeseries_id, observation_date, value
            FROM oil_network.timeseries_data
            ORDER BY timeseries_id, observation_date, saved_date DESC
        ),
        resolved_ts AS (
            SELECT v.variable_id, va.timeseries_id
            FROM oil_network.variables v
            JOIN oil_network.variable_assignments va USING (variable_id)
            WHERE va.timeseries_id IS NOT NULL
            UNION ALL
            SELECT v.variable_id, mirror_va.timeseries_id
            FROM oil_network.variables v
            JOIN oil_network.variable_assignments va USING (variable_id)
            JOIN oil_network.variable_assignments mirror_va
              ON mirror_va.variable_id = va.formula
             AND mirror_va.timeseries_id IS NOT NULL
            WHERE va.timeseries_id IS NULL
              AND va.formula IS NOT NULL
              AND va.formula NOT IN ('0', 'latent()', 'sum_over_children', 'sum_over_outflows')
              AND cardinality(va.formula_inputs) = 1
        ),
        aggregates AS (
            SELECT va.variable_id AS parent_var,
                   va.timeseries_id AS parent_ts,
                   va.formula_inputs AS constituents
            FROM oil_network.variable_assignments va
            WHERE va.timeseries_id IS NOT NULL
              AND cardinality(va.formula_inputs) > 0
        ),
        constituent_values AS (
            SELECT a.parent_var, td.observation_date,
                   SUM(td.value) AS sum_constituents,
                   COUNT(*) AS n_with_data,
                   cardinality(a.constituents) AS n_total
            FROM aggregates a
            CROSS JOIN LATERAL unnest(a.constituents) AS u(child_var_id)
            JOIN resolved_ts rt ON rt.variable_id = u.child_var_id
            JOIN latest td ON td.timeseries_id = rt.timeseries_id
            GROUP BY a.parent_var, a.constituents, td.observation_date
        ),
        observed AS (
            SELECT a.parent_var, td.observation_date, td.value AS observed_value
            FROM aggregates a
            JOIN latest td ON td.timeseries_id = a.parent_ts
        )
        SELECT
            o.parent_var, o.observation_date,
            o.observed_value, c.sum_constituents,
            o.observed_value - c.sum_constituents AS gap,
            CASE WHEN o.observed_value <> 0
                 THEN (o.observed_value - c.sum_constituents) / o.observed_value * 100.0
                 ELSE NULL END AS gap_pct,
            c.n_with_data, c.n_total,
            CASE
              WHEN ABS(o.observed_value - c.sum_constituents) < 1.0
                OR (o.observed_value <> 0
                    AND ABS((o.observed_value - c.sum_constituents) / o.observed_value) < 0.01)
                THEN 'ok'
              WHEN c.n_with_data < c.n_total
                THEN 'partial_coverage'
              ELSE 'inconsistent'
            END AS status
        FROM observed o
        JOIN constituent_values c
          ON c.parent_var = o.parent_var AND c.observation_date = o.observation_date
    """))

print("Step 7 done. 8 views created in oil_network:")
print("  v_flow_edges                   -- physical flow topology (from relational outflow variables)")
print("  v_aggregation_edges            -- aggregation graph (from formula_inputs)")
print("  v_node_production_sources      -- discovery: every node with a production variable + its binding")
print("  v_scope_authoritative_nodes    -- every TS-bound variable_assignments row")
print("  v_scope_collapsed_nodes        -- every '0' or 'latent()' variable")
print("  v_inventory_changes            -- monthly delta inventory (MBBL + kbd) for every TS-bound stock series")
print("  v_aggregate_balance            -- closed mass balance (B = closure residual) at USA + 5 PADDs")
print("  v_aggregation_consistency      -- TS observation vs sum-of-constituents gap (data-quality flag)")

Step 7 done. 8 views created in oil_network:
  v_flow_edges                   -- physical flow topology (from relational outflow variables)
  v_aggregation_edges            -- aggregation graph (from formula_inputs)
  v_node_production_sources      -- discovery: every node with a production variable + its binding
  v_scope_authoritative_nodes    -- every TS-bound variable_assignments row
  v_scope_collapsed_nodes        -- every '0' or 'latent()' variable
  v_inventory_changes            -- monthly delta inventory (MBBL + kbd) for every TS-bound stock series
  v_aggregate_balance            -- closed mass balance (B = closure residual) at USA + 5 PADDs
  v_aggregation_consistency      -- TS observation vs sum-of-constituents gap (data-quality flag)


## Verify

List the tables and their column shapes — a quick sanity check while DBeaver is open on the side.

In [11]:
with engine.connect() as c:
    tables = pd.read_sql(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :s
        ORDER BY table_name
    """), c, params={"s": PG_SCHEMA})
    columns = pd.read_sql(text("""
        SELECT table_name, column_name, data_type, is_nullable, column_default
        FROM information_schema.columns
        WHERE table_schema = :s
        ORDER BY table_name, ordinal_position
    """), c, params={"s": PG_SCHEMA})
    fks = pd.read_sql(text("""
        SELECT tc.table_name, kcu.column_name,
               ccu.table_name  AS references_table,
               ccu.column_name AS references_column
        FROM information_schema.table_constraints tc
        JOIN information_schema.key_column_usage kcu
          ON tc.constraint_name = kcu.constraint_name
         AND tc.table_schema    = kcu.table_schema
        JOIN information_schema.constraint_column_usage ccu
          ON ccu.constraint_name = tc.constraint_name
         AND ccu.table_schema    = tc.table_schema
        WHERE tc.constraint_type = 'FOREIGN KEY'
          AND tc.table_schema = :s
        ORDER BY tc.table_name, kcu.column_name
    """), c, params={"s": PG_SCHEMA})

print(f"Tables in {PG_SCHEMA}:")
display(tables)
print("Columns:")
display(columns)
print("Foreign keys:")
display(fks)

Tables in oil_network:


,table_name
0,assets
1,commodities
2,graphs
3,locations
4,node_type_default_formulas
5,node_types
6,nodes
7,scenario_node_role
8,scenario_notes
9,scenarios


Columns:


,table_name,column_name,data_type,is_nullable,column_default
0,assets,asset_id,text,NO,None
1,assets,name,text,NO,None
2,assets,description,text,YES,None
3,assets,location_id,text,YES,None
4,assets,attributes,jsonb,NO,'{}'::jsonb
...,...,...,...,...,...
153,variables,commodity,text,NO,None
154,variables,node_id,text,NO,None
155,variables,related_node_id,text,YES,None
156,variables,description,text,YES,None


Foreign keys:


,table_name,column_name,references_table,references_column
0,assets,location_id,locations,location_id
1,node_type_default_formulas,commodity,commodities,commodity
2,node_type_default_formulas,node_type,node_types,node_type
3,node_type_default_formulas,variable_type,variable_types,variable_type
4,nodes,asset_id,assets,asset_id
5,nodes,graph_id,graphs,graph_id
6,nodes,node_type,node_types,node_type
7,scenario_node_role,node_id,nodes,node_id
8,scenario_node_role,scenario_id,scenarios,scenario_id
9,scenario_notes,scenario_id,scenarios,scenario_id
